# DBpediaAdapter Example
Demonstrates basic usage of the DBpediaAdapter.

In [4]:
# Add logging to debug DBpediaAdapter failures
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('dbpedia_demo')

try:
    from aid_pais_knowledgegraph.knowledge_lookup.adapters.dbpedia_adapter import DBpediaAdapter
    from aid_pais_knowledgegraph.knowledge_lookup.models import LookupConfig
    config = LookupConfig()
    adapter = DBpediaAdapter(config)
    logger.info('Adapter initialized.')
    results = await adapter.search_concepts('Myalgic encephalomyelitis/chronic fatigue syndrome', limit=5)
    logger.info(f'Results: {results}')
    for concept in results:
        print(concept.primary_id, concept.primary_label)
except Exception as e:
    logger.error(f'Error during DBpedia search: {e}', exc_info=True)

ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7ca0d50e09d0>
INFO:dbpedia_demo:Adapter initialized.
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.dbpedia_adapter:DBpedia API Request URL: https://dbpedia.org/sparql
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.dbpedia_adapter:DBpedia API Request Params: {'query': "\n            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n            PREFIX dbo: <http://dbpedia.org/ontology/>\n            SELECT DISTINCT ?resource ?label ?abstract ?type WHERE {\n              ?resource rdfs:label ?label .\n              FILTER (lang(?label) = 'en')\n              FILTER (regex(?label, 'Myalgic encephalomyelitis/chronic fatigue syndrome', 'i'))\n              OPTIONAL { ?resource dbo:abstract ?abstract . FILTER (lang(?abstract) = 'en') }\n              OPTIONAL { ?resource rdf:type ?type }\n            }\n            \nLIMIT 5", 'format': 'json'}
INFO:dbpedia_demo:Adapter initial

In [ ]:
import SPARQLWrapper


# Improved SPARQL query for DBpedia (browser and API compatible)
# This query uses regex for more robust matching and avoids LCASE for better browser compatibility.
improved_query = '''
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT DISTINCT ?resource ?label ?abstract WHERE {
  ?resource rdfs:label ?label .
  FILTER (lang(?label) = 'en')
  FILTER (regex(?label, 'chronic fatigue syndrome', 'i'))
  OPTIONAL { ?resource dbo:abstract ?abstract . FILTER (lang(?abstract) = 'en') }
}
LIMIT 5
'''
print('Improved SPARQL query:')
print(improved_query)

# You can paste this query directly into the DBpedia SPARQL endpoint browser:
# https://dbpedia.org/sparql

# Run the improved query using SPARQLWrapper
sparql = SPARQLWrapper.SPARQLWrapper("https://dbpedia.org/sparql")
sparql.setQuery(improved_query)
try:
    results = sparql.query().convert()
    print('Improved SPARQLWrapper results:')
    for result in results['results']['bindings']:
        uri = result['resource']['value']
        label = result['label']['value']
        abstract = result.get('abstract', {}).get('value', '')
        print(uri, label)
        if abstract:
            print('  Abstract:', abstract[:120] + ('...' if len(abstract) > 120 else ''))
except Exception as e:
    import logging
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger('sparqlwrapper_demo')
    logger.error(f'Improved SPARQLWrapper error: {e}', exc_info=True)

Improved SPARQL query:

PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT DISTINCT ?resource ?label ?abstract WHERE {
  ?resource rdfs:label ?label .
  FILTER (lang(?label) = 'en')
  FILTER (regex(?label, 'chronic fatigue syndrome', 'i'))
  OPTIONAL { ?resource dbo:abstract ?abstract . FILTER (lang(?abstract) = 'en') }
}
LIMIT 5



NameError: name 'sparql' is not defined

In [ ]:
# Debug: Search for DBpedia categories with label 'Chronic fatigue syndrome' (en)
# This will find category resources with the exact label.
category_query = '''
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?resource ?label WHERE {
  ?resource rdfs:label ?label .
  FILTER (lang(?label) = 'en')
  FILTER (regex(str(?resource), 'Category:Chronic_fatigue_syndrome'))
  FILTER (?label = 'Chronic fatigue syndrome')
}
LIMIT 5
'''
print('Category SPARQL query:')
print(category_query)

sparql.setQuery(category_query)
try:
    results = sparql.query().convert()
    print('Category search results:')
    for result in results['results']['bindings']:
        uri = result['resource']['value']
        label = result['label']['value']
        print(uri, label)
except Exception as e:
    import logging
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger('sparqlwrapper_demo')
    logger.error(f'Category SPARQLWrapper error: {e}', exc_info=True)